# RAG-Augmented Distillation — Kaggle GPU Run (2×T4, 30GB)

Runs the complete RAD pipeline end-to-end on Kaggle's 2×T4 (30GB total VRAM).

**Pipeline:**
1. Build ChromaDB from SQuAD v2 contexts (~15 min, CPU)
2. Generate teacher soft labels — RAG + bare + negative logits (~30 min, GPU)
3. Train RAD student — flan-t5-small with L_RAD loss (~45 min, GPU)
4. Evaluate — bare teacher vs RAG teacher vs RAD student

**Key result to look for:** Student (RAD) EM/F1 should exceed Student trained with standard KD,
demonstrating that RAG-augmented soft labels transfer retrieved knowledge into student weights.

> Enable GPU: Settings → Accelerator → GPU T4 x2

In [ ]:
import torch, subprocess, os

# Verify GPU
if not torch.cuda.is_available():
    raise RuntimeError("No GPU found. Enable GPU in Settings → Accelerator → GPU T4 x2")

n_gpus = torch.cuda.device_count()
for i in range(n_gpus):
    props = torch.cuda.get_device_properties(i)
    print(f"GPU {i}: {props.name}  {props.total_memory/1e9:.1f} GB")
print(f"Total VRAM: {sum(torch.cuda.get_device_properties(i).total_memory for i in range(n_gpus))/1e9:.1f} GB")

In [ ]:
# Clone repo
REPO_DIR = '/kaggle/working/model-distillation'
if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone',
        'https://github.com/aabhimittal/model-distillation-', REPO_DIR], check=True)

os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")

# Install dependencies (quiet)
subprocess.run(['pip', 'install', '-e', '.[dev]', '-q'], check=True)
print("Dependencies installed.")

In [ ]:
# Paths — all under /kaggle/working so they persist for the session
BASE_DIR        = '/kaggle/working/rad'
CHROMA_DIR      = f'{BASE_DIR}/chroma_db'
SOFT_LABELS_DIR = f'{BASE_DIR}/soft_labels'
OUTPUT_DIR      = f'{BASE_DIR}/outputs'

for d in [CHROMA_DIR, SOFT_LABELS_DIR, OUTPUT_DIR]:
    os.makedirs(d, exist_ok=True)

# Use device_map=auto to spread teacher across both T4 GPUs during label generation
DEVICE_MAP = 'auto' if torch.cuda.device_count() > 1 else None
print(f"device_map: {DEVICE_MAP}")
print(f"ChromaDB   : {CHROMA_DIR}")
print(f"Soft labels: {SOFT_LABELS_DIR}")
print(f"Outputs    : {OUTPUT_DIR}")

## Phase 1 — Build ChromaDB vector store

Embeds ~50k unique SQuAD v2 Wikipedia passages into ChromaDB using
`sentence-transformers/all-MiniLM-L6-v2` (runs on CPU, ~15 min).
Idempotent — safe to re-run.

In [ ]:
cmd = [
    'python', 'scripts/build_vector_db.py',
    '--config', 'configs/distillation_config.yaml',
    '--chroma-dir', CHROMA_DIR,
]
subprocess.run(cmd, check=True)

## Phase 2 — Generate teacher soft labels

Runs `flan-t5-base` (teacher) on all 10k training questions:
- **RAG pass**: teacher sees top-3 ChromaDB passages → logits for L_RAG
- **Bare pass**: teacher sees no context → logits for L_KL  
- **Negative pass**: teacher sees irrelevant context → logits for L_CRA

Saves three logit tensors per example as `.npz` files.
With `--device-map auto` the teacher is distributed across both T4 GPUs.

In [ ]:
cmd = [
    'python', 'scripts/generate_soft_labels.py',
    '--config', 'configs/distillation_config.yaml',
    '--soft-labels-dir', SOFT_LABELS_DIR,
    '--chroma-dir', CHROMA_DIR,
]
if DEVICE_MAP:
    cmd += ['--device-map', DEVICE_MAP]

subprocess.run(cmd, check=True)

In [ ]:
# Sanity check: inspect one soft label file
import glob, numpy as np
npz_files = sorted(glob.glob(f'{SOFT_LABELS_DIR}/*.npz'))
print(f"Soft label files: {len(npz_files)}")

sample = np.load(npz_files[0])
print(f"Keys: {list(sample.keys())}")
for k in sample.keys():
    print(f"  {k}: shape={sample[k].shape}, dtype={sample[k].dtype}")

# Verify RAG and bare logits differ (if identical, retrieval has no effect)
kl = np.mean(np.sum(
    np.exp(sample['rag_logits']) * (sample['rag_logits'] - sample['bare_logits']), axis=-1
))
print(f"\nMean KL(RAG || bare) for this example: {kl:.4f}")
print("(> 0 means retrieval shifted the teacher's distribution — good!)")

## Phase 3 — Train the RAD student

Trains `flan-t5-small` with the full L_RAD loss for 3 epochs (10k examples).

```
L_RAD = 0.5·L_RAG + 0.2·L_KL + 0.1·L_CRA + 0.2·L_CE
```

Checkpoints saved every 500 steps. Loss components logged every 50 steps.

In [ ]:
cmd = [
    'python', 'scripts/train_student.py',
    '--config', 'configs/distillation_config.yaml',
    '--output-dir', OUTPUT_DIR,
    '--soft-labels-dir', SOFT_LABELS_DIR,
    '--chroma-dir', CHROMA_DIR,
]
# Note: student stays on a single GPU during training (gradient flow requires it)
subprocess.run(cmd, check=True)

In [ ]:
# Plot all loss components over training steps
import json
import matplotlib.pyplot as plt

with open(f'{OUTPUT_DIR}/loss_history.json') as f:
    history = json.load(f)

steps = [h['step'] for h in history]

fig, axes = plt.subplots(1, 2, figsize=(15, 4))

axes[0].plot(steps, [h['loss'] for h in history], color='black', linewidth=1.5)
axes[0].set_title('Total RAD Loss')
axes[0].set_xlabel('Global Step')
axes[0].set_ylabel('Loss')
axes[0].grid(alpha=0.3)

component_colors = {
    'L_RAG': ('L_RAG — student vs RAG-teacher (novel)', 'tab:blue'),
    'L_KL':  ('L_KL  — student vs bare teacher (std KD)', 'tab:orange'),
    'L_CRA': ('L_CRA — contrastive retrieval alignment', 'tab:green'),
    'L_CE':  ('L_CE  — hard label cross-entropy', 'tab:red'),
}
for key, (label, color) in component_colors.items():
    axes[1].plot(steps, [h[key] for h in history], label=label, color=color, linewidth=1.2)
axes[1].set_title('Loss Components')
axes[1].set_xlabel('Global Step')
axes[1].legend(fontsize=8)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/loss_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {OUTPUT_DIR}/loss_curves.png")

## Phase 4 — Evaluate: compare all model conditions

Evaluates three conditions on SQuAD v2 validation set (1k examples):
1. **Teacher (bare)** — flan-t5-base, no retrieval
2. **Teacher + RAG** — flan-t5-base with ChromaDB retrieval (oracle upper bound)
3. **Student (RAD)** — flan-t5-small, no retrieval at inference

**Expected result**: Student (RAD) EM/F1 > a standard-KD student (trained with L_KL only)

In [ ]:
STUDENT_CHECKPOINT = f'{OUTPUT_DIR}/final'

cmd = [
    'python', 'scripts/evaluate.py',
    '--config', 'configs/distillation_config.yaml',
    '--student-rad', STUDENT_CHECKPOINT,
    '--output-dir', OUTPUT_DIR,
    '--chroma-dir', CHROMA_DIR,
]
subprocess.run(cmd, check=True)

In [ ]:
# Final results table + bar chart
import json
import numpy as np
import matplotlib.pyplot as plt

with open(f'{OUTPUT_DIR}/eval_results.json') as f:
    results = json.load(f)

# ── Table ──
print(f"\n{'='*62}")
print(f"  RAG-AUGMENTED DISTILLATION — RESULTS (SQuAD v2)")
print(f"{'='*62}")
print(f"{'Model':<30} {'EM':>8} {'F1':>8} {'BLEU-4':>8}")
print(f"{'-'*62}")
for model_name, metrics in results.items():
    em = metrics.get('exact_match', 0) * 100
    f1 = metrics.get('f1', 0) * 100
    b4 = metrics.get('bleu4', 0) * 100
    print(f"{model_name:<30} {em:>7.1f}%  {f1:>7.1f}%  {b4:>7.1f}%")
print(f"{'='*62}\n")

# ── Bar chart ──
models    = list(results.keys())
em_scores = [results[m].get('exact_match', 0) * 100 for m in models]
f1_scores = [results[m].get('f1', 0) * 100 for m in models]

x     = np.arange(len(models))
width = 0.35
colors_em = ['#4C72B0', '#DD8452', '#55A868']
colors_f1 = ['#6F9DC8', '#E8A87C', '#77C28A']

fig, ax = plt.subplots(figsize=(11, 5))
bars1 = ax.bar(x - width/2, em_scores, width, label='Exact Match (%)', color=colors_em)
bars2 = ax.bar(x + width/2, f1_scores,  width, label='F1 (%)',          color=colors_f1)

ax.set_xticks(x)
ax.set_xticklabels(models, rotation=10, ha='right', fontsize=11)
ax.set_ylabel('Score (%)', fontsize=12)
ax.set_title('RAG-Augmented Distillation — Model Comparison (SQuAD v2)', fontsize=13)
ax.legend(fontsize=11)
ax.bar_label(bars1, fmt='%.1f%%', padding=3, fontsize=9)
ax.bar_label(bars2, fmt='%.1f%%', padding=3, fontsize=9)
ax.set_ylim(0, max(f1_scores) * 1.25)
ax.grid(axis='y', alpha=0.3)
ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/results_chart.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Chart saved: {OUTPUT_DIR}/results_chart.png")

In [ ]:
# List all output files
import os
print("Output files:")
for root, dirs, files in os.walk(OUTPUT_DIR):
    dirs[:] = [d for d in dirs if not d.startswith('checkpoint')]  # skip checkpoint dirs
    for f in files:
        path = os.path.join(root, f)
        size_mb = os.path.getsize(path) / 1e6
        print(f"  {path}  ({size_mb:.1f} MB)")